# 01 · `gl_engine/config.py`

## What this file is for

Thirty lines, and every one of them is a decision somebody had to make.

It answers two questions: **where does ISO's content live**, and **what is this engine not willing to do**. The second is the interesting half — the engine refuses to rate below a certain date, because below it the corpus can't produce all 51 jurisdictions, and a partial answer that looks complete is worse than no answer.

**Depends on:** nothing. Everything else depends on this.

## Its public surface

Generated from the module, so it can't drift from the code.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import inspect
from gl_engine import config

for name, obj in vars(config).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != config.__name__:
        continue
    if inspect.isclass(obj):
        print(f"class {name}")
        for m, f in vars(obj).items():
            if m.startswith("_"):
                continue
            if isinstance(f, property):
                print(f"    .{m}  (property)")
            elif callable(f):
                print(f"    .{m}{inspect.signature(f)}")
    elif inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        print(f"{name} = {obj!r}")

## The smallest thing that works

Where is the corpus, and is it actually there?

In [ ]:
from gl_engine import config

root = config.CORPUS_ROOT
print("corpus root :", root)
print("exists      :", root.exists())

if root.exists():
    states = sorted(d.name for d in root.iterdir()
                    if d.is_dir() and d.name not in config.EXCLUDE_DIRS)
    print("directories :", len(states))
    print(" ", ", ".join(states[:12]), "...")

The path is a default, not a hard-coding — set `GL_ERC_ROOT` in your environment and this points somewhere else. That is the whole mechanism by which the engine reads a licensed corpus that is deliberately **not** in this repository.

## The interesting case

### The floor, and why it exists

`MIN_ASOF` is the earliest date the engine will accept. It is not a stylistic choice — below it, the corpus cannot resolve every jurisdiction.

In [ ]:
print("MIN_ASOF          :", config.MIN_ASOF)
print("CLASS_BASIS_CLIFF :", config.CLASS_BASIS_CLIFF)
print("COUNTRYWIDE       :", config.COUNTRYWIDE)
print("EXCLUDE_DIRS      :", set(config.EXCLUDE_DIRS))

from gl_engine import EditionResolver
r = EditionResolver()

ok = r.resolve_all(config.MIN_ASOF)
print(f"\nat the floor exactly: all {len(ok)} jurisdictions resolve")

### Why `EXCLUDE_DIRS` is not cosmetic

`_quarantine_misfiled` holds a byte-identical duplicate package that was set aside during analysis. Counting it would double-count a jurisdiction — and every measurement in this project is stated as *n of N*, so a wrong N quietly corrupts a lot of documents.

In [ ]:
import itertools

excluded = [d.name for d in config.CORPUS_ROOT.iterdir()
            if d.is_dir() and d.name in config.EXCLUDE_DIRS]
print("excluded directories:", excluded or "(none present)")

for d in excluded:
    inside = list(itertools.islice((config.CORPUS_ROOT / d).rglob("*"), 4))
    print(f"  {d} holds e.g. {[p.name for p in inside]}")

## What it refuses

One day below the floor is the difference between an answer and a refusal.

In [ ]:
from gl_engine.errors import ResolutionError

for date in (config.MIN_ASOF, "20220831"):
    try:
        n = len(r.resolve_all(date))
        print(f"{date}: resolves all {n}")
    except ResolutionError as e:
        print(f"{date}: REFUSED\n    {e}")

## Try it yourself

1. Point `GL_ERC_ROOT` at a directory that doesn't exist. What fails, and how early?
2. How many jurisdictions resolve one day *below* `MIN_ASOF`? Is the floor tight, or conservative?
3. `CLASS_BASIS_CLIFF` is a date. Find what changes on it — [`13-schema-fields`](13-schema-fields.ipynb) is the notebook that uses it.

In [ ]:
# your turn